In [1]:
import pandas as pd
import numpy as np

ml_data = pd.read_csv(
    "../data/processed/karhut_ml_dataset.csv",
    parse_dates=["date"]
)

print(ml_data.shape)
print(ml_data["date"].min())
print(ml_data["date"].max())

(34272, 39)
2026-05-08 00:00:00
2026-09-10 00:00:00


In [2]:
fire_features = [
    "fire_count_lag1",
    "fire_count_lag3",
    "fire_count_lag7",
    "frp_sum_lag1",
    "frp_sum_lag3",
    "frp_sum_lag7",
    "fire_count_roll3",
    "fire_count_roll7",
    "frp_sum_roll3",
    "frp_sum_roll7"
]

weather_features = [
    "temperature_mean",
    "temperature_max",
    "relative_humidity_mean",
    "precipitation_sum",
    "boundary_layer_height_mean",
    "vapour_pressure_deficit_mean",
    "wind_speed_mean",
    "wind_speed_max",
    "wind_u_mean",
    "wind_v_mean"
]

features_20 = fire_features + weather_features

target = "target_fire_active_t1"

In [3]:
print(len(features_20))

20


In [4]:
cells = (
    ml_data[["cell_id", "lat", "lon"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Number of cells:", len(cells))
print("Unique lat:", cells["lat"].nunique())
print("Unique lon:", cells["lon"].nunique())

print("\nLatitude values:")
print(np.sort(cells["lat"].unique()))

print("\nLongitude values:")
print(np.sort(cells["lon"].unique()))

Number of cells: 272
Unique lat: 17
Unique lon: 16

Latitude values:
[-4.5  -3.75 -3.   -2.25 -1.5  -0.75  0.    0.75  1.5   2.25  3.    3.75
  4.5   5.25  6.    6.75  7.5 ]

Longitude values:
[108.   108.75 109.5  110.25 111.   111.75 112.5  113.25 114.   114.75
 115.5  116.25 117.   117.75 118.5  119.25]


In [5]:
lat_values = np.sort(cells["lat"].unique())
lon_values = np.sort(cells["lon"].unique())

print("Latitude spacing:")
print(np.unique(np.round(np.diff(lat_values), 5)))

print("\nLongitude spacing:")
print(np.unique(np.round(np.diff(lon_values), 5)))

Latitude spacing:
[0.75]

Longitude spacing:
[0.75]


In [7]:
# ============================================================
# STEP 4 — Build Spatial Neighbor Dictionary
# Queen adjacency (8-neighbor)
# ============================================================

neighbor_dict = {}

for _, cell in cells.iterrows():

    cell_id = cell["cell_id"]
    lat = cell["lat"]
    lon = cell["lon"]

    # Selisih koordinat terhadap semua cell
    dlat = np.abs(cells["lat"] - lat)
    dlon = np.abs(cells["lon"] - lon)

    # Queen adjacency:
    # Δlat <= 0.75 dan Δlon <= 0.75
    # tetapi bukan cell itu sendiri
    neighbors = cells[
        (dlat <= 0.75) &
        (dlon <= 0.75) &
        ((dlat > 0) | (dlon > 0))
    ]

    neighbor_dict[cell_id] = neighbors["cell_id"].tolist()

print("Number of cells:", len(neighbor_dict))

Number of cells: 272


In [8]:
neighbor_counts = pd.Series({
    cell_id: len(neighbors)
    for cell_id, neighbors in neighbor_dict.items()
})

print(neighbor_counts.describe())
print("\nNeighbor count distribution:")
print(neighbor_counts.value_counts().sort_index())

count    272.000000
mean       7.286765
std        1.335897
min        3.000000
25%        8.000000
50%        8.000000
75%        8.000000
max        8.000000
dtype: float64

Neighbor count distribution:
3      4
5     58
8    210
Name: count, dtype: int64


In [9]:
print("Number of cells:", len(neighbor_dict))

Number of cells: 272


In [10]:
print(neighbor_counts.value_counts().sort_index())

3      4
5     58
8    210
Name: count, dtype: int64


In [11]:
# ============================================================
# STEP 5A — Prepare Fire History Lookup
# ============================================================

fire_history = ml_data[
    [
        "date",
        "cell_id",
        "fire_count_lag1",
        "frp_sum_lag1"
    ]
].copy()

print(fire_history.shape)
print(fire_history.head())

(34272, 4)
        date      cell_id  fire_count_lag1  frp_sum_lag1
0 2026-05-08  -0.75_108.0              0.0           0.0
1 2026-05-09  -0.75_108.0              0.0           0.0
2 2026-05-10  -0.75_108.0              0.0           0.0
3 2026-05-11  -0.75_108.0              0.0           0.0
4 2026-05-12  -0.75_108.0              0.0           0.0


In [12]:
# ============================================================
# STEP 5B — Calculate Spatial-Lag Features
# ============================================================

spatial_rows = []

for _, row in ml_data.iterrows():

    cell_id = row["cell_id"]
    date = row["date"]

    neighbors = neighbor_dict[cell_id]

    neighbor_data = fire_history[
        (fire_history["date"] == date) &
        (fire_history["cell_id"].isin(neighbors))
    ]

    # Rata-rata fire count dari tetangga
    neighbor_fire_count = neighbor_data[
        "fire_count_lag1"
    ].mean()

    # Rasio tetangga yang fire-active
    # fire_count_lag1 > 0 berarti cell tersebut aktif
    neighbor_fire_ratio = (
        neighbor_data["fire_count_lag1"] > 0
    ).mean()

    # Rata-rata FRP dari tetangga
    neighbor_frp_mean = neighbor_data[
        "frp_sum_lag1"
    ].mean()

    spatial_rows.append({
        "date": date,
        "cell_id": cell_id,
        "neighbor_fire_count_lag1":
            neighbor_fire_count,

        "neighbor_fire_active_ratio_lag1":
            neighbor_fire_ratio,

        "neighbor_frp_mean_lag1":
            neighbor_frp_mean
    })

spatial_features_df = pd.DataFrame(spatial_rows)

print(spatial_features_df.shape)
spatial_features_df.head()

(34272, 5)


,date,cell_id,neighbor_fire_count_lag1,neighbor_fire_active_ratio_lag1,neighbor_frp_mean_lag1
0,2026-05-08,-0.75_108.0,0.0,0.0,0.0
1,2026-05-09,-0.75_108.0,0.0,0.0,0.0
2,2026-05-10,-0.75_108.0,0.0,0.0,0.0
3,2026-05-11,-0.75_108.0,0.0,0.0,0.0
4,2026-05-12,-0.75_108.0,0.0,0.0,0.0


In [13]:
ml_spatial = ml_data.merge(
    spatial_features_df,
    on=["date", "cell_id"],
    how="left",
    validate="one_to_one"
)

print(ml_spatial.shape)

(34272, 42)


In [15]:
spatial_cols = [
    "neighbor_fire_count_lag1",
    "neighbor_fire_active_ratio_lag1",
    "neighbor_frp_mean_lag1"
]

In [16]:
print(
    ml_spatial[spatial_cols].isna().sum()
)

neighbor_fire_count_lag1           0
neighbor_fire_active_ratio_lag1    0
neighbor_frp_mean_lag1             0
dtype: int64


In [17]:
print(
    ml_spatial[spatial_cols].describe()
)

       neighbor_fire_count_lag1  neighbor_fire_active_ratio_lag1  \
count              34272.000000                     34272.000000   
mean                   4.071525                         0.181750   
std                   17.574515                         0.262485   
min                    0.000000                         0.000000   
25%                    0.000000                         0.000000   
50%                    0.000000                         0.000000   
75%                    0.800000                         0.250000   
max                  385.625000                         1.000000   

       neighbor_frp_mean_lag1  
count            34272.000000  
mean                32.666461  
std                149.285258  
min                  0.000000  
25%                  0.000000  
50%                  0.000000  
75%                  3.961438  
max               3724.923750  


In [18]:
print(
    ml_spatial[
        "neighbor_fire_active_ratio_lag1"
    ].value_counts().sort_index()
)

neighbor_fire_active_ratio_lag1
0.000000    17901
0.125000     4536
0.200000     1106
0.250000     2967
0.333333       30
0.375000     2035
0.400000      296
0.500000     1584
0.600000      110
0.625000     1098
0.666667        2
0.750000      970
0.800000       20
0.875000      789
1.000000      828
Name: count, dtype: int64


In [19]:
# ============================================================
# STEP 6 — Spatial Feature vs Target
# ============================================================

ml_spatial["neighbor_ratio_bin"] = pd.cut(
    ml_spatial["neighbor_fire_active_ratio_lag1"],
    bins=[-0.001, 0, 0.25, 0.5, 0.75, 1.0],
    labels=[
        "0%",
        ">0–25%",
        ">25–50%",
        ">50–75%",
        ">75–100%"
    ]
)

spatial_target_analysis = (
    ml_spatial
    .groupby("neighbor_ratio_bin", observed=True)
    .agg(
        samples=(target, "size"),
        fire_rate=(target, "mean")
    )
)

spatial_target_analysis["fire_rate_percent"] = (
    spatial_target_analysis["fire_rate"] * 100
)

spatial_target_analysis

,samples,fire_rate,fire_rate_percent
neighbor_ratio_bin,,,
0%,17901,0.052734,5.273448
>0–25%,8609,0.140667,14.066674
>25–50%,3945,0.332573,33.257288
>50–75%,2180,0.649083,64.908257
>75–100%,1637,0.855223,85.522297


In [20]:
print(
    ml_spatial[
        [
            "neighbor_fire_count_lag1",
            "neighbor_fire_active_ratio_lag1",
            "neighbor_frp_mean_lag1",
            target
        ]
    ].corr()[target]
)

neighbor_fire_count_lag1           0.368271
neighbor_fire_active_ratio_lag1    0.560378
neighbor_frp_mean_lag1             0.345746
target_fire_active_t1              1.000000
Name: target_fire_active_t1, dtype: float64


In [22]:
# ============================================================
# STEP 6 — Time-Based Train / Validation / Test Split
# Same split as previous experiments
# ============================================================

train = ml_spatial[
    ml_spatial["date"] < "2026-08-01"
].copy()

val = ml_spatial[
    (ml_spatial["date"] >= "2026-08-01") &
    (ml_spatial["date"] < "2026-09-01")
].copy()

test = ml_spatial[
    ml_spatial["date"] >= "2026-09-01"
].copy()

print("Train:", train.shape)
print("Validation:", val.shape)
print("Test:", test.shape)

print("\nDate ranges:")
print("Train:", train["date"].min(), "→", train["date"].max())
print("Validation:", val["date"].min(), "→", val["date"].max())
print("Test:", test["date"].min(), "→", test["date"].max())

print("\nTarget distribution:")
print("Train:")
print(train[target].value_counts())

print("\nValidation:")
print(val[target].value_counts())

print("\nTest:")
print(test[target].value_counts())

Train: (23120, 43)
Validation: (8432, 43)
Test: (2720, 43)

Date ranges:
Train: 2026-05-08 00:00:00 → 2026-07-31 00:00:00
Validation: 2026-08-01 00:00:00 → 2026-08-31 00:00:00
Test: 2026-09-01 00:00:00 → 2026-09-10 00:00:00

Target distribution:
Train:
target_fire_active_t1
0.0    20567
1.0     2553
Name: count, dtype: int64

Validation:
target_fire_active_t1
0.0    5572
1.0    2860
Name: count, dtype: int64

Test:
target_fire_active_t1
0.0    1851
1.0     869
Name: count, dtype: int64


In [23]:
# ============================================================
# EXPERIMENT B1
# Logistic Regression + Spatial Ratio
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

features_B1 = features_20 + [
    "neighbor_fire_active_ratio_lag1"
]

model_B1 = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

model_B1.fit(
    train[features_B1],
    train[target]
)

val_prob_B1 = model_B1.predict_proba(
    val[features_B1]
)[:, 1]

val_pred_B1 = (
    val_prob_B1 >= 0.5
).astype(int)

print("=== B1: LR + Spatial Ratio ===")

print(
    "PR-AUC:",
    average_precision_score(
        val[target],
        val_prob_B1
    )
)

print(
    "Precision:",
    precision_score(
        val[target],
        val_pred_B1
    )
)

print(
    "Recall:",
    recall_score(
        val[target],
        val_pred_B1
    )
)

print(
    "F1:",
    f1_score(
        val[target],
        val_pred_B1
    )
)

print(
    "Accuracy:",
    accuracy_score(
        val[target],
        val_pred_B1
    )
)

=== B1: LR + Spatial Ratio ===
PR-AUC: 0.8973164686899754
Precision: 0.6375059269796112
Recall: 0.9402097902097902
F1: 0.7598191579542244
Accuracy: 0.7983870967741935


In [24]:
# ============================================================
# EXPERIMENT B2
# Logistic Regression + 20 Features + 3 Spatial Features
# ============================================================

features_B2 = features_20 + [
    "neighbor_fire_count_lag1",
    "neighbor_fire_active_ratio_lag1",
    "neighbor_frp_mean_lag1"
]

model_B2 = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

model_B2.fit(
    train[features_B2],
    train[target]
)

val_prob_B2 = model_B2.predict_proba(
    val[features_B2]
)[:, 1]

val_pred_B2 = (
    val_prob_B2 >= 0.5
).astype(int)

print("=== B2: LR + 3 Spatial Features ===")

print(
    "PR-AUC:",
    average_precision_score(
        val[target],
        val_prob_B2
    )
)

print(
    "Precision:",
    precision_score(
        val[target],
        val_pred_B2
    )
)

print(
    "Recall:",
    recall_score(
        val[target],
        val_pred_B2
    )
)

print(
    "F1:",
    f1_score(
        val[target],
        val_pred_B2
    )
)

print(
    "Accuracy:",
    accuracy_score(
        val[target],
        val_pred_B2
    )
)

=== B2: LR + 3 Spatial Features ===
PR-AUC: 0.8873152754924274
Precision: 0.6262482168330956
Recall: 0.920979020979021
F1: 0.7455420322671951
Accuracy: 0.7867647058823529


In [25]:
# ============================================================
# EXPERIMENT C
# XGBoost + 20 Benchmark Features
# ============================================================

from xgboost import XGBClassifier

# Calculate class weight from TRAIN only
neg = (train[target] == 0).sum()
pos = (train[target] == 1).sum()

scale_pos_weight = neg / pos

print("scale_pos_weight:", scale_pos_weight)

model_C = XGBClassifier(
    n_estimators=400,
    max_depth=3,
    learning_rate=0.03,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

model_C.fit(
    train[features_20],
    train[target]
)

val_prob_C = model_C.predict_proba(
    val[features_20]
)[:, 1]

val_pred_C = (
    val_prob_C >= 0.5
).astype(int)

print("\n=== C: XGBoost + 20 Features ===")

print(
    "PR-AUC:",
    average_precision_score(
        val[target],
        val_prob_C
    )
)

print(
    "Precision:",
    precision_score(
        val[target],
        val_pred_C
    )
)

print(
    "Recall:",
    recall_score(
        val[target],
        val_pred_C
    )
)

print(
    "F1:",
    f1_score(
        val[target],
        val_pred_C
    )
)

print(
    "Accuracy:",
    accuracy_score(
        val[target],
        val_pred_C
    )
)

scale_pos_weight: 8.056012534273403

=== C: XGBoost + 20 Features ===
PR-AUC: 0.8909525351067774
Precision: 0.6839469070874029
Recall: 0.9548951048951049
F1: 0.7970232015175835
Accuracy: 0.8350332068311196


In [26]:
# ============================================================
# EXPERIMENT D
# XGBoost + 20 Features + Spatial Features
# ============================================================

features_D = features_20 + [
    "neighbor_fire_count_lag1",
    "neighbor_fire_active_ratio_lag1",
    "neighbor_frp_mean_lag1"
]

model_D = XGBClassifier(
    n_estimators=400,
    max_depth=3,
    learning_rate=0.03,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

model_D.fit(
    train[features_D],
    train[target]
)

val_prob_D = model_D.predict_proba(
    val[features_D]
)[:, 1]

val_pred_D = (
    val_prob_D >= 0.5
).astype(int)

print("\n=== D: XGBoost + Spatial ===")

print(
    "PR-AUC:",
    average_precision_score(
        val[target],
        val_prob_D
    )
)

print(
    "Precision:",
    precision_score(
        val[target],
        val_pred_D
    )
)

print(
    "Recall:",
    recall_score(
        val[target],
        val_pred_D
    )
)

print(
    "F1:",
    f1_score(
        val[target],
        val_pred_D
    )
)

print(
    "Accuracy:",
    accuracy_score(
        val[target],
        val_pred_D
    )
)


=== D: XGBoost + Spatial ===
PR-AUC: 0.873299279664127
Precision: 0.6587643678160919
Recall: 0.9618881118881119
F1: 0.7819783968163729
Accuracy: 0.8180740037950665


In [27]:
# ============================================================
# D1 — XGBoost + Spatial Ratio Only
# ============================================================

features_D1 = features_20 + [
    "neighbor_fire_active_ratio_lag1"
]

model_D1 = XGBClassifier(
    n_estimators=400,
    max_depth=3,
    learning_rate=0.03,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

model_D1.fit(
    train[features_D1],
    train[target]
)

val_prob_D1 = model_D1.predict_proba(
    val[features_D1]
)[:, 1]

val_pred_D1 = (
    val_prob_D1 >= 0.5
).astype(int)

print("=== D1: XGBoost + Spatial Ratio ===")

print(
    "PR-AUC:",
    average_precision_score(
        val[target],
        val_prob_D1
    )
)

print(
    "Precision:",
    precision_score(
        val[target],
        val_pred_D1
    )
)

print(
    "Recall:",
    recall_score(
        val[target],
        val_pred_D1
    )
)

print(
    "F1:",
    f1_score(
        val[target],
        val_pred_D1
    )
)

print(
    "Accuracy:",
    accuracy_score(
        val[target],
        val_pred_D1
    )
)

=== D1: XGBoost + Spatial Ratio ===
PR-AUC: 0.8765024156313004
Precision: 0.6647328982354364
Recall: 0.9615384615384616
F1: 0.7860511647849078
Accuracy: 0.8224620493358634


In [28]:
# ============================================================
# Feature Importance — XGBoost + Spatial Ratio
# ============================================================

importance_D1 = pd.Series(
    model_D1.feature_importances_,
    index=features_D1
).sort_values(ascending=False)

print(importance_D1.head(20))

frp_sum_roll7                      0.310932
fire_count_roll7                   0.298248
neighbor_fire_active_ratio_lag1    0.053204
frp_sum_lag1                       0.043534
precipitation_sum                  0.033017
boundary_layer_height_mean         0.027335
temperature_max                    0.025309
temperature_mean                   0.023247
fire_count_roll3                   0.022389
fire_count_lag1                    0.020440
wind_u_mean                        0.017139
relative_humidity_mean             0.017076
wind_speed_mean                    0.016851
wind_speed_max                     0.015753
frp_sum_lag7                       0.014990
vapour_pressure_deficit_mean       0.014581
frp_sum_roll3                      0.010004
wind_v_mean                        0.009941
fire_count_lag7                    0.009576
frp_sum_lag3                       0.009501
dtype: float32


In [29]:
importance_D = pd.Series(
    model_D.feature_importances_,
    index=features_D
).sort_values(ascending=False)

print(importance_D.head(20))

frp_sum_roll7                      0.321562
fire_count_roll7                   0.228529
frp_sum_roll3                      0.049665
neighbor_fire_active_ratio_lag1    0.045046
frp_sum_lag1                       0.033941
neighbor_frp_mean_lag1             0.032675
frp_sum_lag7                       0.031209
precipitation_sum                  0.030140
boundary_layer_height_mean         0.025333
neighbor_fire_count_lag1           0.023121
temperature_max                    0.022567
temperature_mean                   0.020586
fire_count_roll3                   0.017570
wind_speed_max                     0.015026
fire_count_lag1                    0.014854
wind_u_mean                        0.014785
wind_speed_mean                    0.014348
vapour_pressure_deficit_mean       0.014145
relative_humidity_mean             0.013262
wind_v_mean                        0.009531
dtype: float32


In [8]:
[name for name, obj in globals().copy().items()
 if isinstance(obj, pd.DataFrame)]

['df', 'model_df', 'X', 'X_train', 'X_val', 'X_test']

In [9]:
print(model_df.shape)
print(model_df.columns.tolist())

(34272, 23)
['date', 'cell_id', 'fire_count_lag1', 'fire_count_lag3', 'fire_count_lag7', 'frp_sum_lag1', 'frp_sum_lag3', 'frp_sum_lag7', 'fire_count_roll3', 'fire_count_roll7', 'frp_sum_roll3', 'frp_sum_roll7', 'temperature_mean', 'temperature_max', 'relative_humidity_mean', 'precipitation_sum', 'boundary_layer_height_mean', 'vapour_pressure_deficit_mean', 'wind_speed_mean', 'wind_speed_max', 'wind_u_mean', 'wind_v_mean', 'target_fire_active_t1']


In [10]:
fire_full = pd.read_csv(
    "../data/intermediate/fire_kalimantan_cell_day_full.csv"
)

print(fire_full.shape)
print(fire_full.columns.tolist())
print(fire_full.head())

(6524, 6)
['cell_id', 'date', 'n_detections', 'frp_sum', 'frp_mean', 'frp_max']
        cell_id        date  n_detections  frp_sum  frp_mean  frp_max
0   -0.75_117.0  2026-05-01             1     1.01  1.010000     1.01
1  -0.75_117.75  2026-05-01             1     0.33  0.330000     0.33
2    -1.5_117.0  2026-05-01             3     6.31  2.103333     2.65
3   -2.25_115.5  2026-05-01             5     5.90  1.180000     1.67
4    -3.0_115.5  2026-05-01             2     3.35  1.675000     1.96


In [12]:
import numpy as np
import pandas as pd

# Copy data yang sudah ada
spatial_data = model_df.copy()

# Pastikan tanggal bertipe datetime
spatial_data["date"] = pd.to_datetime(spatial_data["date"])

# Load fire cell-day aktif
fire_full = pd.read_csv(
    "../data/intermediate/fire_kalimantan_cell_day_full.csv"
)

fire_full["date"] = pd.to_datetime(fire_full["date"])

print("ML data:", spatial_data.shape)
print("Fire active data:", fire_full.shape)

ML data: (34272, 23)
Fire active data: (6524, 6)


In [13]:
def parse_cell_id(cell_id):
    lat, lon = cell_id.rsplit("_", 1)
    return float(lat), float(lon)

coords = spatial_data[["cell_id"]].drop_duplicates().copy()

coords[["lat", "lon"]] = coords["cell_id"].apply(
    lambda x: pd.Series(parse_cell_id(x))
)

print(coords.shape)
print(coords.head())

(272, 3)
          cell_id   lat     lon
0     -0.75_108.0 -0.75  108.00
126  -0.75_108.75 -0.75  108.75
252   -0.75_109.5 -0.75  109.50
378  -0.75_110.25 -0.75  110.25
504   -0.75_111.0 -0.75  111.00


In [14]:
cells = coords["cell_id"].tolist()

cell_coords = {
    row["cell_id"]: (row["lat"], row["lon"])
    for _, row in coords.iterrows()
}

neighbors = {}

for cell_id, (lat, lon) in cell_coords.items():
    cell_neighbors = []

    for other_id, (other_lat, other_lon) in cell_coords.items():
        if cell_id == other_id:
            continue

        lat_diff = abs(lat - other_lat)
        lon_diff = abs(lon - other_lon)

        # Queen adjacency:
        # horizontal, vertical, dan diagonal
        if lat_diff <= 0.75 and lon_diff <= 0.75:
            cell_neighbors.append(other_id)

    neighbors[cell_id] = cell_neighbors

neighbor_counts = pd.Series({
    cell: len(nbs)
    for cell, nbs in neighbors.items()
})

print("Jumlah cell:", len(neighbors))
print("\nDistribusi jumlah tetangga:")
print(neighbor_counts.value_counts().sort_index())

Jumlah cell: 272

Distribusi jumlah tetangga:
3      4
5     58
8    210
Name: count, dtype: int64


In [15]:
active_lookup = set(
    zip(
        fire_full["date"],
        fire_full["cell_id"]
    )
)

print("Jumlah active cell-days:", len(active_lookup))

Jumlah active cell-days: 6524


In [16]:
spatial_ratios = []

for _, row in spatial_data[["date", "cell_id"]].iterrows():

    current_date = row["date"]
    cell_id = row["cell_id"]

    previous_date = current_date - pd.Timedelta(days=1)

    cell_neighbors = neighbors[cell_id]

    if len(cell_neighbors) == 0:
        ratio = np.nan
    else:
        active_neighbors = sum(
            (previous_date, neighbor_id) in active_lookup
            for neighbor_id in cell_neighbors
        )

        ratio = active_neighbors / len(cell_neighbors)

    spatial_ratios.append(ratio)

spatial_data["neighbor_fire_active_ratio_lag1"] = spatial_ratios

In [17]:
print(
    spatial_data["neighbor_fire_active_ratio_lag1"].describe()
)

print(
    "\nMissing:",
    spatial_data["neighbor_fire_active_ratio_lag1"].isna().sum()
)

count    34272.000000
mean         0.181750
std          0.262485
min          0.000000
25%          0.000000
50%          0.000000
75%          0.250000
max          1.000000
Name: neighbor_fire_active_ratio_lag1, dtype: float64

Missing: 0


In [18]:
print(
    spatial_data[
        [
            "neighbor_fire_active_ratio_lag1",
            "target_fire_active_t1"
        ]
    ].corr()
)

                                 neighbor_fire_active_ratio_lag1  \
neighbor_fire_active_ratio_lag1                         1.000000   
target_fire_active_t1                                   0.560378   

                                 target_fire_active_t1  
neighbor_fire_active_ratio_lag1               0.560378  
target_fire_active_t1                         1.000000  


In [19]:
bins = pd.cut(
    spatial_data["neighbor_fire_active_ratio_lag1"],
    bins=[-0.001, 0, 0.25, 0.50, 0.75, 1.0],
    labels=[
        "0%",
        ">0–25%",
        ">25–50%",
        ">50–75%",
        ">75–100%"
    ]
)

ratio_summary = spatial_data.groupby(
    bins,
    observed=False
)["target_fire_active_t1"].agg(
    samples="count",
    fire_rate="mean"
)

ratio_summary["fire_rate"] *= 100

display(ratio_summary)

,samples,fire_rate
neighbor_fire_active_ratio_lag1,,
0%,17901,5.273448
>0–25%,8609,14.066674
>25–50%,3945,33.257288
>50–75%,2180,64.908257
>75–100%,1637,85.522297


In [20]:
corr_features = [
    "fire_count_roll7",
    "fire_count_lag1",
    "frp_sum_roll7",
    "frp_sum_lag1"
]

pearson = spatial_data[
    ["neighbor_fire_active_ratio_lag1"] + corr_features
].corr()["neighbor_fire_active_ratio_lag1"].drop(
    "neighbor_fire_active_ratio_lag1"
).sort_values(ascending=False)

print("=== Pearson Correlation ===")
display(pearson)

=== Pearson Correlation ===


frp_sum_roll7       0.373319
fire_count_roll7    0.372914
fire_count_lag1     0.355038
frp_sum_lag1        0.325631
Name: neighbor_fire_active_ratio_lag1, dtype: float64

In [21]:
spearman = spatial_data[
    ["neighbor_fire_active_ratio_lag1"] + corr_features
].corr(
    method="spearman"
)["neighbor_fire_active_ratio_lag1"].drop(
    "neighbor_fire_active_ratio_lag1"
).sort_values(ascending=False)

print("=== Spearman Correlation ===")
display(spearman)

=== Spearman Correlation ===


fire_count_lag1     0.587532
frp_sum_lag1        0.587301
fire_count_roll7    0.572797
frp_sum_roll7       0.570174
Name: neighbor_fire_active_ratio_lag1, dtype: float64

In [22]:
# Pastikan date sudah datetime
spatial_data["date"] = pd.to_datetime(spatial_data["date"])

features_20 = [
    "fire_count_lag1",
    "fire_count_lag3",
    "fire_count_lag7",
    "frp_sum_lag1",
    "frp_sum_lag3",
    "frp_sum_lag7",
    "fire_count_roll3",
    "fire_count_roll7",
    "frp_sum_roll3",
    "frp_sum_roll7",
    "temperature_mean",
    "temperature_max",
    "relative_humidity_mean",
    "precipitation_sum",
    "boundary_layer_height_mean",
    "vapour_pressure_deficit_mean",
    "wind_speed_mean",
    "wind_speed_max",
    "wind_u_mean",
    "wind_v_mean"
]

spatial_feature = "neighbor_fire_active_ratio_lag1"
target = "target_fire_active_t1"

train = spatial_data[spatial_data["date"] < "2026-08-01"].copy()

val = spatial_data[
    (spatial_data["date"] >= "2026-08-01") &
    (spatial_data["date"] < "2026-09-01")
].copy()

test = spatial_data[
    spatial_data["date"] >= "2026-09-01"
].copy()

print("Train:", train.shape)
print("Validation:", val.shape)
print("Test:", test.shape)

Train: (23120, 24)
Validation: (8432, 24)
Test: (2720, 24)


In [23]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix
)

features_B1 = features_20 + [spatial_feature]

X_train = train[features_B1]
y_train = train[target].astype(int)

X_test = test[features_B1]
y_test = test[target].astype(int)

lr_b1 = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

lr_b1.fit(X_train, y_train)

# Probability
test_proba_b1 = lr_b1.predict_proba(X_test)[:, 1]

# Threshold 0.5
test_pred_b1 = (test_proba_b1 >= 0.5).astype(int)

print("=== B1 — LR + Spatial Ratio ===")
print(f"PR-AUC   : {average_precision_score(y_test, test_proba_b1):.6f}")
print(f"Precision: {precision_score(y_test, test_pred_b1):.6f}")
print(f"Recall   : {recall_score(y_test, test_pred_b1):.6f}")
print(f"F1       : {f1_score(y_test, test_pred_b1):.6f}")
print(f"Accuracy : {accuracy_score(y_test, test_pred_b1):.6f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_pred_b1))

=== B1 — LR + Spatial Ratio ===
PR-AUC   : 0.897698
Precision: 0.581072
Recall   : 0.960875
F1       : 0.724198
Accuracy : 0.766176

Confusion Matrix:
[[1249  602]
 [  34  835]]


In [24]:
from xgboost import XGBClassifier

# Class imbalance dihitung dari TRAIN saja
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

scale_pos_weight = neg / pos

print("scale_pos_weight:", scale_pos_weight)

xgb_d1 = XGBClassifier(
    n_estimators=400,
    max_depth=3,
    learning_rate=0.03,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_d1.fit(X_train, y_train)

# Probability
test_proba_d1 = xgb_d1.predict_proba(X_test)[:, 1]

# Threshold 0.5
test_pred_d1 = (test_proba_d1 >= 0.5).astype(int)

print("=== D1 — XGBoost + Spatial Ratio ===")
print(f"PR-AUC   : {average_precision_score(y_test, test_proba_d1):.6f}")
print(f"Precision: {precision_score(y_test, test_pred_d1):.6f}")
print(f"Recall   : {recall_score(y_test, test_pred_d1):.6f}")
print(f"F1       : {f1_score(y_test, test_pred_d1):.6f}")
print(f"Accuracy : {accuracy_score(y_test, test_pred_d1):.6f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_pred_d1))

scale_pos_weight: 8.056012534273403
=== D1 — XGBoost + Spatial Ratio ===
PR-AUC   : 0.892469
Precision: 0.594803
Recall   : 0.974684
F1       : 0.738770
Accuracy : 0.779779

Confusion Matrix:
[[1274  577]
 [  22  847]]


In [26]:
print(ml_spatial.columns.tolist())

NameError: name 'ml_spatial' is not defined